# Notebook 09: Inventory & Reorder Intelligence (Store CA_1)
## Step 1: Import Libraries and Verify Uploaded Files

### What we are doing in this cell
We are importing the Python libraries needed for inventory calculations and model loading. We are also defining the exact paths for all 7 manually uploaded files in Google Colab and checking that every file exists.

### Why we need this step
Verifying all files right at the start ensures we have both forecasting models (H7 and H28), validation datasets, predictions, feature lists, and item groups ready before we run any inventory formulas.

### What problem this step solves
It prevents `FileNotFoundError` crashes later in the notebook and keeps all file paths organized in one place.

### What output we should expect
A checklist showing `[OK]` and the file size (in MB/KB) for all 7 uploaded files.

In [ ]:
# Standard Python libraries for files, JSON, and math
import os
import json
import math
import warnings

# Libraries for data handling and numerical calculations
import numpy as np
import pandas as pd

# Library for loading our pre-trained LightGBM models
import joblib

# Ignore minor warning messages to keep output clean
warnings.filterwarnings("ignore")

# Define paths for all 7 manually uploaded files in Google Colab
MODEL_H7_PATH = "/content/lightgbm_h7.joblib"
MODEL_H28_PATH = "/content/lightgbm_h28.joblib"
VAL_H7_PATH = "/content/ca1_h7_val.parquet"
VAL_H28_PATH = "/content/ca1_h28_val.parquet"
FEATURES_PATH = "/content/feature_lists.json"
H7_VAL_PREDS_PATH = "/content/h7_val_predictions.parquet"
ITEM_GROUPS_PATH = "/content/item_groups.csv"

# Put all paths into a dictionary to verify them cleanly
files_to_check = {
    "H7 LightGBM Model": MODEL_H7_PATH,
    "H28 LightGBM Model": MODEL_H28_PATH,
    "H7 Validation Data": VAL_H7_PATH,
    "H28 Validation Data": VAL_H28_PATH,
    "Feature Lists JSON": FEATURES_PATH,
    "H7 Validation Predictions": H7_VAL_PREDS_PATH,
    "Item Groups CSV": ITEM_GROUPS_PATH
}

print("Checking uploaded files in Google Colab...\n")

all_files_found = True

for label, path in files_to_check.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"[OK] {label:<26}: {path} ({size_mb:.2f} MB)")
    else:
        print(f"[MISSING] {label:<21}: {path} NOT FOUND!")
        all_files_found = False

print("\nStatus:", "All 7 files are ready!" if all_files_found else "Some files are missing. Please check file names.")

Checking uploaded files in Google Colab...

[OK] H7 LightGBM Model         : /content/lightgbm_h7.joblib (1.74 MB)
[OK] H28 LightGBM Model        : /content/lightgbm_h28.joblib (8.33 MB)
[OK] H7 Validation Data        : /content/ca1_h7_val.parquet (0.22 MB)
[OK] H28 Validation Data       : /content/ca1_h28_val.parquet (0.23 MB)
[OK] Feature Lists JSON        : /content/feature_lists.json (0.06 MB)
[OK] H7 Validation Predictions : /content/h7_val_predictions.parquet (0.78 MB)
[OK] Item Groups CSV           : /content/item_groups.csv (0.30 MB)

Status: All 7 files are ready!


## Step 2: Load Models and Datasets & Inspect Columns

### What we are doing in this cell
We are loading both pre-trained LightGBM models (`H7` and `H28`), `feature_lists.json`, `h7_val_predictions.parquet`, `item_groups.csv`, and both validation datasets (`ca1_h7_val.parquet` and `ca1_h28_val.parquet`). Then, we inspect their shapes, column names, date ranges, and first few rows.

### Why we need this step
Before calculating typical daily forecast error (`item_error_spread.csv`) or building our reorder engine (`reorder.py`), we must inspect the exact column names inside `h7_val_predictions.parquet` and `ca1_h28_val.parquet` so we do not assume anything.

### What problem this step solves
It prevents `KeyError` issues caused by guessing column names (for example, checking whether predictions are named `forecast`, `prediction`, or `y_pred`) and confirms that all 3,049 items are present.

### What output we should expect
Confirmation of both loaded LightGBM models, the shapes and column lists of all loaded tables, date ranges, and a preview of the first 5 rows of `h7_val_predictions.parquet` and `item_groups.csv`.

In [ ]:
# 1. Load both pre-trained LightGBM models (NO retraining)
model_h7 = joblib.load(MODEL_H7_PATH)
model_h28 = joblib.load(MODEL_H28_PATH)

print("H7 Model Type: ", type(model_h7))
print("H28 Model Type:", type(model_h28))

# 2. Load feature_lists.json
with open(FEATURES_PATH, "r") as file:
    feature_lists = json.load(file)

print("\nKeys in feature_lists.json:", list(feature_lists.keys()))

# 3. Load validation predictions, item groups, and validation datasets
h7_val_preds = pd.read_parquet(H7_VAL_PREDS_PATH)
item_groups = pd.read_csv(ITEM_GROUPS_PATH)
val_h7_df = pd.read_parquet(VAL_H7_PATH)
val_h28_df = pd.read_parquet(VAL_H28_PATH)

# 4. Inspect shapes and column names
print(f"\nh7_val_predictions Shape: {h7_val_preds.shape[0]:,} rows, {h7_val_preds.shape[1]} columns")
print("h7_val_predictions Columns:", list(h7_val_preds.columns))

print(f"\nitem_groups Shape:        {item_groups.shape[0]:,} rows, {item_groups.shape[1]} columns")
print("item_groups Columns:", list(item_groups.columns))

print(f"\nval_h7_df Shape:          {val_h7_df.shape[0]:,} rows, {val_h7_df.shape[1]} columns")
print(f"val_h28_df Shape:         {val_h28_df.shape[0]:,} rows, {val_h28_df.shape[1]} columns")
print("val_h28_df Columns:", list(val_h28_df.columns))

# 5. Check date range in h7_val_predictions (if 'date' column exists)
if "date" in h7_val_preds.columns:
    print(f"\nh7_val_predictions Date Range: {h7_val_preds['date'].min()} to {h7_val_preds['date'].max()}")

# 6. Check missing values in h7_val_predictions and item_groups
print("\nMissing values in h7_val_predictions:", h7_val_preds.isnull().sum().sum())
print("Missing values in item_groups:       ", item_groups.isnull().sum().sum())

# 7. Preview first 5 rows of h7_val_predictions and item_groups
print("\nPreview of h7_val_predictions:")
print(h7_val_preds.head())

print("\nPreview of item_groups:")
print(item_groups.head())

H7 Model Type:  <class 'lightgbm.sklearn.LGBMRegressor'>
H28 Model Type: <class 'lightgbm.sklearn.LGBMRegressor'>

Keys in feature_lists.json: ['features_7', 'features_28', 'item_categories']

h7_val_predictions Shape: 85,372 rows, 4 columns
h7_val_predictions Columns: ['item_id', 'date', 'sales', 'forecast']

item_groups Shape:        3,049 rows, 6 columns
item_groups Columns: ['item_id', 'avg_sales', 'zero_sales_share', 'sales_std', 'weekend_sales_ratio', 'group']

val_h7_df Shape:          85,372 rows, 26 columns
val_h28_df Shape:         85,359 rows, 26 columns
val_h28_df Columns: ['item_id', 'store_id', 'date', 'sales', 'lag_28', 'rolling_mean_28', 'wday', 'month', 'year', 'is_weekend', 'snap', 'has_event', 'sell_price', 'price_change', 'ev_Cultural', 'ev_National', 'ev_Religious', 'ev_Sporting', 'ev_None', 'dept_FOODS_1', 'dept_FOODS_2', 'dept_FOODS_3', 'dept_HOBBIES_1', 'dept_HOBBIES_2', 'dept_HOUSEHOLD_1', 'dept_HOUSEHOLD_2']

h7_val_predictions Date Range: 2016-03-28 00:00:00 

## Step 3: Calculate Typical Daily Forecast Error (`item_error_spread`)

### What we are doing in this cell
We are calculating the daily forecast error (`error = sales - forecast`) for every row in `h7_val_predictions`. Then, we compute the **standard deviation** of these errors for each of the 3,049 items. If any item has insufficient validation rows (fewer than 7 days) or a missing standard deviation, we automatically fall back to its **item group's average error**.

### Why we need this step
Forecasts are never 100% perfect. To calculate how much **Safety Stock** a store needs to avoid running out of an item, we need to know how much the forecast typically fluctuates for that specific item.

### What problem this step solves
It creates a reliable, item-level measure of forecast uncertainty (`typical_daily_error`) for all 3,049 items and safely handles edge cases where an item might have too few validation observations.

### What output we should expect
The total number of items processed (3,049), how many items needed the group-average fallback, the average typical daily error for each item group, and a preview of the first 5 rows of `item_error_spread`.

In [ ]:
# 1. Calculate daily forecast error on validation predictions
h7_val_preds["error"] = h7_val_preds["sales"] - h7_val_preds["forecast"]

# 2. Group by item_id to count validation rows and calculate error standard deviation
item_error_stats = (
    h7_val_preds.groupby("item_id", observed=True)["error"]
    .agg(val_rows="count", raw_error_std="std")
    .reset_index()
)

# 3. Merge with item_groups so all 3,049 items have their K-Means group label
item_error_spread = item_groups[["item_id", "group"]].merge(
    item_error_stats,
    on="item_id",
    how="left"
)

# Fill missing row counts with 0 (in case an item had no validation rows)
item_error_spread["val_rows"] = item_error_spread["val_rows"].fillna(0).astype(int)

# 4. Calculate each group's average error std (using items with at least 7 validation days)
MIN_VAL_ROWS = 7
valid_items = item_error_spread[item_error_spread["val_rows"] >= MIN_VAL_ROWS]
group_avg_error = valid_items.groupby("group")["raw_error_std"].mean()

print("--- Average Daily Forecast Error by Item Group ---")
for grp, avg_err in group_avg_error.items():
    print(f" - {grp:<15}: {avg_err:.4f} units/day")

# 5. Apply fallback rule: if val_rows < 7 or std is NaN, use the item's group average
needs_fallback = (item_error_spread["val_rows"] < MIN_VAL_ROWS) | (item_error_spread["raw_error_std"].isna())

item_error_spread["used_group_fallback"] = needs_fallback
item_error_spread["typical_daily_error"] = np.where(
    needs_fallback,
    item_error_spread["group"].map(group_avg_error),
    item_error_spread["raw_error_std"]
)

# 6. Print summary and preview the final table
print(f"\nTotal Items in item_error_spread: {len(item_error_spread):,}")
print(f"Items using group-average fallback: {needs_fallback.sum():,}")
print(f"Missing values in typical_daily_error: {item_error_spread['typical_daily_error'].isna().sum()}")

print("\nPreview of item_error_spread (First 5 items):")
print(item_error_spread.head())

--- Average Daily Forecast Error by Item Group ---
 - Fast Sellers   : 7.0805 units/day
 - Medium Sellers : 2.1052 units/day
 - Slow Sellers   : 0.7967 units/day

Total Items in item_error_spread: 3,049
Items using group-average fallback: 0
Missing values in typical_daily_error: 0

Preview of item_error_spread (First 5 items):
       item_id         group  val_rows  raw_error_std  used_group_fallback  \
0  FOODS_1_001  Slow Sellers        28       1.290134                False   
1  FOODS_1_002  Slow Sellers        28       0.806307                False   
2  FOODS_1_003  Slow Sellers        28       1.057344                False   
3  FOODS_1_004  Fast Sellers        28       0.821892                False   
4  FOODS_1_005  Slow Sellers        28       1.207497                False   

   typical_daily_error  
0             1.290134  
1             0.806307  
2             1.057344  
3             0.821892  
4             1.207497  


## Step 4: Generate H28 Validation Forecasts (For Lead Times > 7 Days)

### What we are doing in this cell
We are preparing `val_h28_df` by recreating the 4 cyclical time features (`wday_sin`, `wday_cos`, `month_sin`, `month_cos`) and setting `item_id` to the exact categorical list from `feature_lists.json`. Then, we use our pre-trained `model_h28` to generate 28-day horizon forecasts and store them in a clean table (`h28_val_preds`) that matches the format of `h7_val_preds`.

### Why we need this step
According to our DemandLens methodology, we use **H7 forecasts** when a supplier's lead time is 7 days or less, and **H28 forecasts** when the lead time is longer than 7 days. We already have `h7_val_predictions.parquet`, so generating `h28_val_preds` gives us both forecast horizons ready for our reorder engine.

### What problem this step solves
It ensures our reorder calculations can accurately handle both short lead times (e.g., 3–7 days) and longer lead times (e.g., 10–21 days) without retraining any model.

### What output we should expect
Feature verification confirmation for `model_h28`, the H28 validation WAPE, the shape of `h28_val_preds`, and a preview of the first 5 rows.

In [ ]:
# 1. Recreate the 4 cyclical time features on val_h28_df
val_h28_df["wday_sin"] = np.sin(2 * np.pi * val_h28_df["wday"] / 7)
val_h28_df["wday_cos"] = np.cos(2 * np.pi * val_h28_df["wday"] / 7)
val_h28_df["month_sin"] = np.sin(2 * np.pi * val_h28_df["month"] / 12)
val_h28_df["month_cos"] = np.cos(2 * np.pi * val_h28_df["month"] / 12)

# 2. Ensure 'item_id' uses the exact category list from feature_lists.json
val_h28_df["item_id"] = pd.Categorical(
    val_h28_df["item_id"],
    categories=feature_lists["item_categories"]
)

# 3. Verify all features expected by model_h28 are present
features_28 = list(model_h28.feature_name_)
missing_h28 = [col for col in features_28 if col not in val_h28_df.columns]

if missing_h28:
    print(f"[ERROR] Missing features for model_h28: {missing_h28}")
else:
    print(f"[OK] All {len(features_28)} required features for model_h28 are present.")

    # 4. Generate H28 forecasts (NO retraining) and clip negative values to 0
    print("Generating H28 validation forecasts...")
    raw_h28_preds = model_h28.predict(val_h28_df[features_28])
    val_h28_df["forecast"] = np.clip(raw_h28_preds, 0, None)

    # 5. Create a clean table matching h7_val_preds columns
    h28_val_preds = val_h28_df[["item_id", "date", "sales", "forecast"]].copy()

    # 6. Check H28 validation WAPE
    h28_abs_err = np.abs(h28_val_preds["sales"] - h28_val_preds["forecast"]).sum()
    h28_wape = (h28_abs_err / h28_val_preds["sales"].sum()) * 100

    print(f"\nH28 Validation WAPE: {h28_wape:.2f}%")
    print(f"h28_val_preds Shape: {h28_val_preds.shape[0]:,} rows, {h28_val_preds.shape[1]} columns")

    print("\nPreview of h28_val_preds (First 5 rows):")
    print(h28_val_preds.head())

[OK] All 27 required features for model_h28 are present.
Generating H28 validation forecasts...

H28 Validation WAPE: 74.33%
h28_val_preds Shape: 85,359 rows, 4 columns

Preview of h28_val_preds (First 5 rows):
       item_id       date  sales  forecast
0  FOODS_1_001 2016-03-28      2  0.648588
1  FOODS_1_001 2016-03-29      1  0.557860
2  FOODS_1_001 2016-03-30      1  0.641618
3  FOODS_1_001 2016-03-31      0  0.668558
4  FOODS_1_001 2016-04-01      4  0.680063


## Step 5: Build the Core Inventory & Reorder Calculation Function

### What we are doing in this cell
We are writing a clean, reusable Python function called `calculate_reorder_plan()` that takes user inputs (`item_id`, `current_stock`, and `lead_time_days`) and calculates all 6 required inventory metrics:
1. **Expected Demand in Lead Time** (using **H7** forecasts if `lead_time_days <= 7`, and **H28** forecasts if `lead_time_days > 7`)
2. **Safety Stock** = `1.65 × typical_daily_error × sqrt(lead_time_days)`
3. **Reorder Point** = `expected_demand_in_lead_time + safety_stock`
4. **Days of Stock Left** = `current_stock / average_daily_forecast`
5. **Order Quantity** = `expected demand over (lead_time_days + 7 review days) + safety stock - current stock` (minimum `0`, rounded up to whole units)
6. **Stockout Risk Level**:
   - **High**: stock lasts less than `lead_time_days`
   - **Medium**: stock lasts less than `lead_time_days + 3` days
   - **Low**: otherwise

### Why we need this step
Putting all inventory math into one well-structured function ensures consistent calculations across our notebook, testing scenarios, and the `reorder.py` module we will export for the DemandLens backend.

### What problem this step solves
It converts raw daily sales forecasts into clear, actionable reorder decisions (how much to order and how urgent the risk is) while safely handling edge cases like zero forecasted demand.

### What output we should expect
A confirmation message that `calculate_reorder_plan()` has been defined and is ready to test.

In [ ]:
def calculate_reorder_plan(
    item_id,
    current_stock,
    lead_time_days,
    h7_preds_df,
    h28_preds_df,
    error_spread_df,
    review_days=7
):
    """
    Calculates inventory and reorder intelligence for a single item
    based on user-provided current_stock and lead_time_days.
    """
    # 1. Choose the right forecast horizon based on lead time
    # Use H7 for lead_time <= 7 days, and H28 for longer lead times
    if lead_time_days <= 7:
        forecast_source = "H7"
        item_forecasts = h7_preds_df[h7_preds_df["item_id"] == item_id].sort_values("date")
    else:
        forecast_source = "H28"
        item_forecasts = h28_preds_df[h28_preds_df["item_id"] == item_id].sort_values("date")

    if len(item_forecasts) == 0:
        raise ValueError(f"Item '{item_id}' not found in {forecast_source} forecast data.")

    # Extract daily forecast values as a simple list
    daily_forecasts = item_forecasts["forecast"].values

    # 2. Expected demand in lead time (sum of next lead_time_days forecasts)
    lead_time_slice = daily_forecasts[:lead_time_days]
    expected_demand_lead_time = float(np.sum(lead_time_slice))

    # Average daily forecast during the lead time
    avg_daily_forecast = float(np.mean(lead_time_slice))

    # 3. Look up typical_daily_error and group from item_error_spread
    item_error_row = error_spread_df[error_spread_df["item_id"] == item_id]
    if len(item_error_row) == 0:
        raise ValueError(f"Item '{item_id}' not found in item_error_spread.")

    typical_daily_error = float(item_error_row["typical_daily_error"].values[0])
    item_group = str(item_error_row["group"].values[0])

    # 4. Safety stock = 1.65 * typical_daily_error * sqrt(lead_time_days)
    safety_stock = 1.65 * typical_daily_error * math.sqrt(lead_time_days)

    # 5. Reorder point = expected_demand_in_lead_time + safety_stock
    reorder_point = expected_demand_lead_time + safety_stock

    # 6. Days of stock left = current_stock / average_daily_forecast
    if avg_daily_forecast > 0:
        days_of_stock_left = current_stock / avg_daily_forecast
    else:
        # Avoid dividing by zero if forecasted demand is 0
        days_of_stock_left = 999.0 if current_stock > 0 else 0.0

    # 7. Order quantity:
    # expected demand over (lead_time_days + 7 review days) + safety stock - current stock
    total_window_days = lead_time_days + review_days

    if len(daily_forecasts) >= total_window_days:
        expected_demand_window = float(np.sum(daily_forecasts[:total_window_days]))
    else:
        # If forecast array is shorter than total_window_days, extend using avg_daily_forecast
        missing_days = total_window_days - len(daily_forecasts)
        expected_demand_window = float(np.sum(daily_forecasts)) + (missing_days * avg_daily_forecast)

    raw_order_qty = expected_demand_window + safety_stock - current_stock
    order_quantity = max(0, math.ceil(raw_order_qty))

    # 8. Risk classification
    if days_of_stock_left < lead_time_days:
        risk = "High"
    elif days_of_stock_left < (lead_time_days + 3):
        risk = "Medium"
    else:
        risk = "Low"

    # Return a clean dictionary with all calculated results
    return {
        "item_id": item_id,
        "group": item_group,
        "forecast_model_used": forecast_source,
        "current_stock": current_stock,
        "lead_time_days": lead_time_days,
        "average_daily_forecast": round(avg_daily_forecast, 2),
        "expected_demand_in_lead_time": round(expected_demand_lead_time, 2),
        "typical_daily_error": round(typical_daily_error, 2),
        "safety_stock": round(safety_stock, 2),
        "reorder_point": round(reorder_point, 2),
        "days_of_stock_left": round(days_of_stock_left, 1),
        "order_quantity": int(order_quantity),
        "risk": risk
    }

print("[OK] Function 'calculate_reorder_plan()' is defined and ready to test!")

[OK] Function 'calculate_reorder_plan()' is defined and ready to test!


## Step 6: Test the Reorder Calculation Function on Sample Scenarios

### What we are doing in this cell
We are testing our `calculate_reorder_plan()` function on four example user-input scenarios covering:
- **Fast Sellers**, **Medium Sellers**, and **Slow Sellers**
- **Short lead times** (`<= 7` days, which should trigger the **H7** model)
- **Long lead times** (`> 7` days, which should trigger the **H28** model)
- Different stock levels to verify **High**, **Medium**, and **Low** risk classifications

*(Note: These stock numbers and lead times are example user inputs for testing only, not real store inventory.)*

### Why we need this step
Before exporting `reorder.py` for the DemandLens backend, we must verify that every formula—especially safety stock, order quantity rounding, model switching (`H7` vs `H28`), and risk classification—behaves exactly as expected.

### What problem this step solves
It catches any logic bugs, division-by-zero issues, or unexpected values early while we are still inside the notebook.

### What output we should expect
A clean summary table showing the calculated reorder plan for all four test scenarios, including which forecast model was used (`H7` or `H28`), safety stock, reorder point, days of stock left, order quantity, and risk level.

In [ ]:
# 1. Pick one sample item from each K-Means group
fast_item = item_error_spread[item_error_spread["group"] == "Fast Sellers"]["item_id"].iloc[0]
medium_item = item_error_spread[item_error_spread["group"] == "Medium Sellers"]["item_id"].iloc[0]
slow_item = item_error_spread[item_error_spread["group"] == "Slow Sellers"]["item_id"].iloc[0]

# 2. Define 4 example user-input scenarios to test all edge cases
# (current_stock and lead_time_days are simulated user inputs for testing)
test_scenarios = [
    {"item_id": fast_item,   "current_stock": 10, "lead_time_days": 5},   # Short lead time (H7), low stock
    {"item_id": medium_item, "current_stock": 20, "lead_time_days": 7},   # Boundary lead time (H7)
    {"item_id": slow_item,   "current_stock": 50, "lead_time_days": 10},  # Long lead time (H28), high stock
    {"item_id": fast_item,   "current_stock": 40, "lead_time_days": 14},  # Long lead time (H28), fast seller
]

# 3. Run calculate_reorder_plan() for each scenario
results_list = []

for scenario in test_scenarios:
    plan = calculate_reorder_plan(
        item_id=scenario["item_id"],
        current_stock=scenario["current_stock"],
        lead_time_days=scenario["lead_time_days"],
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )
    results_list.append(plan)

# 4. Display all test results in a clean DataFrame
test_results_df = pd.DataFrame(results_list)

print("--- Reorder Intelligence Test Scenarios (Example User Inputs) ---\n")
print(test_results_df.to_string(index=False))

--- Reorder Intelligence Test Scenarios (Example User Inputs) ---

    item_id          group forecast_model_used  current_stock  lead_time_days  average_daily_forecast  expected_demand_in_lead_time  typical_daily_error  safety_stock  reorder_point  days_of_stock_left  order_quantity risk
FOODS_1_004   Fast Sellers                  H7             10               5                    2.26                         11.32                 0.82          3.03          14.35                 4.4              26 High
FOODS_1_006 Medium Sellers                  H7             20               7                    1.40                          9.83                 1.75          7.62          17.45                14.2               8  Low
FOODS_1_001   Slow Sellers                 H28             50              10                    0.64                          6.43                 1.29          6.73          13.16                77.8               0  Low
FOODS_1_004   Fast Sellers               

## Step 7: Export `item_error_spread.csv` and `reorder.py` to Google Drive

### What we are doing in this cell
We are mounting Google Drive and saving the two required Notebook 09 deliverables into the DemandLens `backend_handover` folder:
1. `item_error_spread.csv`: Contains the typical daily forecast error and item group for all 3,049 items.
2. `reorder.py`: A clean, standalone Python module containing our `calculate_reorder_plan()` function so it can be imported directly by any script or backend later.

### Why we need this step
Google Colab's local storage is temporary. Saving both files to Google Drive ensures they are permanently stored alongside our Notebook 08 exports and ready for future integration.

### What problem this step solves
It packages our tested inventory math into a reusable Python file (`reorder.py`) instead of leaving it trapped inside a notebook cell.

### What output we should expect
Google Drive mount confirmation, followed by `[SAVED]` messages with the file sizes for `item_error_spread.csv` and `reorder.py`.

In [ ]:

# 1. Set Colab Files panel folder
import os
import math
import numpy as np

HANDOVER_DIR = "/content/backend_handover"
os.makedirs(HANDOVER_DIR, exist_ok=True)

# 2. Export item_error_spread.csv
error_spread_path = os.path.join(
    HANDOVER_DIR,
    "item_error_spread.csv"
)

item_error_spread.to_csv(error_spread_path, index=False)


# 3. Create standalone reorder.py module
reorder_module_code = '''"""
DemandLens — Inventory & Reorder Intelligence Module (Store CA_1)
Generated from Notebook 09.
"""

import math
import numpy as np


def calculate_reorder_plan(
    item_id,
    current_stock,
    lead_time_days,
    h7_preds_df,
    h28_preds_df,
    error_spread_df,
    review_days=7
):
    """Calculate inventory and reorder intelligence."""

    # 1. Select forecast based on lead time
    if lead_time_days <= 7:
        forecast_source = "H7"
        item_forecasts = h7_preds_df[
            h7_preds_df["item_id"] == item_id
        ].sort_values("date")
    else:
        forecast_source = "H28"
        item_forecasts = h28_preds_df[
            h28_preds_df["item_id"] == item_id
        ].sort_values("date")

    if len(item_forecasts) == 0:
        raise ValueError(
            f"Item '{item_id}' not found in {forecast_source} forecast data."
        )

    daily_forecasts = item_forecasts["forecast"].values

    # 2. Expected demand and average daily forecast
    lead_time_slice = daily_forecasts[:lead_time_days]

    expected_demand_lead_time = float(np.sum(lead_time_slice))
    avg_daily_forecast = float(np.mean(lead_time_slice))

    # 3. Get item error information
    item_error_row = error_spread_df[
        error_spread_df["item_id"] == item_id
    ]

    if len(item_error_row) == 0:
        raise ValueError(
            f"Item '{item_id}' not found in item_error_spread."
        )

    typical_daily_error = float(
        item_error_row["typical_daily_error"].values[0]
    )

    item_group = str(item_error_row["group"].values[0])

    # 4. Calculate safety stock
    safety_stock = (
        1.65 * typical_daily_error * math.sqrt(lead_time_days)
    )

    # 5. Calculate reorder point
    reorder_point = expected_demand_lead_time + safety_stock

    # 6. Calculate days of stock left
    if avg_daily_forecast > 0:
        days_of_stock_left = current_stock / avg_daily_forecast
    else:
        days_of_stock_left = 999.0 if current_stock > 0 else 0.0

    # 7. Calculate order quantity
    total_window_days = lead_time_days + review_days

    if len(daily_forecasts) >= total_window_days:
        expected_demand_window = float(
            np.sum(daily_forecasts[:total_window_days])
        )
    else:
        missing_days = total_window_days - len(daily_forecasts)

        expected_demand_window = (
            float(np.sum(daily_forecasts))
            + missing_days * avg_daily_forecast
        )

    raw_order_qty = (
        expected_demand_window + safety_stock - current_stock
    )

    order_quantity = max(0, math.ceil(raw_order_qty))

    # 8. Risk classification
    if days_of_stock_left < lead_time_days:
        risk = "High"
    elif days_of_stock_left < lead_time_days + 3:
        risk = "Medium"
    else:
        risk = "Low"

    return {
        "item_id": item_id,
        "group": item_group,
        "forecast_model_used": forecast_source,
        "current_stock": current_stock,
        "lead_time_days": lead_time_days,
        "average_daily_forecast": round(avg_daily_forecast, 2),
        "expected_demand_in_lead_time": round(expected_demand_lead_time, 2),
        "typical_daily_error": round(typical_daily_error, 2),
        "safety_stock": round(safety_stock, 2),
        "reorder_point": round(reorder_point, 2),
        "days_of_stock_left": round(days_of_stock_left, 1),
        "order_quantity": int(order_quantity),
        "risk": risk
    }
'''

reorder_py_path = os.path.join(HANDOVER_DIR, "reorder.py")

with open(reorder_py_path, "w") as f:
    f.write(reorder_module_code)


# 4. Verify exported files
print(f"Saving outputs to: {HANDOVER_DIR}\n")

print("--- Export Verification ---")

for filepath in [error_spread_path, reorder_py_path]:
    size_kb = os.path.getsize(filepath) / 1024

    print(
        f"[SAVED] {os.path.basename(filepath):<25} "
        f"({size_kb:,.1f} KB)"
    )

print("\nAll files exported successfully!")


Saving outputs to: /content/backend_handover

--- Export Verification ---
[SAVED] item_error_spread.csv     (221.3 KB)
[SAVED] reorder.py                (3.4 KB)

All files exported successfully!


## Step 8: Fix `reorder.py` (Add `reorder_advice()` & Input Validation) and Run Verification Tests

### What we are doing in this cell
We are updating `reorder.py` inside our Google Drive `backend_handover` folder to:
1. Expose the required `reorder_advice()` function (while keeping `calculate_reorder_plan` as a backward-compatible alias).
2. Add strict input validation that rejects `current_stock < 0`, `lead_time_days <= 0`, and missing or invalid `item_id` values.
3. Preserve all inventory formulas from `DemandLens_Remaining_Work_Guide.pdf` without modifying `item_error_spread.csv` or any other file.

Then, we import `reorder_advice()` directly from Google Drive and run 6 verification tests: normal reorder case, zero stock, zero demand, negative stock, zero lead time, and invalid `item_id`.

### Why we need this step
Our final verification caught two issues: the function name did not match `reorder_advice()`, and negative stock or zero lead time inputs were not explicitly blocked. Fixing both ensures `reorder.py` is safe, reliable, and ready for backend integration.

### What problem this step solves
It prevents `ImportError` when importing `reorder_advice()`, stops invalid user inputs from causing `NaN` or division errors, and confirms all edge cases behave cleanly.

### What output we should expect
Confirmation that `reorder.py` was updated and imported successfully, followed by `[PASS]` results for all 6 verification test cases.

In [ ]:

import os
import sys
import importlib
import pandas as pd

# 1. Colab Files panel folder
HANDOVER_DIR = "/content/backend_handover"
os.makedirs(HANDOVER_DIR, exist_ok=True)

reorder_py_path = os.path.join(HANDOVER_DIR, "reorder.py")


# 2. Updated reorder.py module
updated_reorder_code = '''"""
DemandLens — Inventory & Reorder Intelligence Module (Store CA_1)
Generated and verified in Notebook 09.
"""

import math
import numpy as np


def reorder_advice(
    item_id,
    current_stock,
    lead_time_days,
    h7_preds_df,
    h28_preds_df,
    error_spread_df,
    review_days=7
):
    """
    Calculates inventory and reorder intelligence for a single item.
    """

    # --- Input Validation ---
    if not isinstance(item_id, str) or not item_id.strip():
        raise ValueError("Invalid item_id: must be a non-empty string.")

    if current_stock is None or current_stock < 0:
        raise ValueError(
            f"Invalid current_stock ({current_stock}): must be >= 0."
        )

    if lead_time_days is None or lead_time_days <= 0:
        raise ValueError(
            f"Invalid lead_time_days ({lead_time_days}): must be > 0."
        )

    # 1. Select H7 or H28 forecast
    if lead_time_days <= 7:
        forecast_source = "H7"
        item_forecasts = h7_preds_df[
            h7_preds_df["item_id"] == item_id
        ].sort_values("date")
    else:
        forecast_source = "H28"
        item_forecasts = h28_preds_df[
            h28_preds_df["item_id"] == item_id
        ].sort_values("date")

    if len(item_forecasts) == 0:
        raise ValueError(
            f"Invalid item_id: '{item_id}' not found in {forecast_source} forecast data."
        )

    daily_forecasts = item_forecasts["forecast"].values

    # 2. Expected demand in lead time
    lead_time_slice = daily_forecasts[:lead_time_days]

    expected_demand_lead_time = float(np.sum(lead_time_slice))
    avg_daily_forecast = float(np.mean(lead_time_slice))

    # 3. Get item error information
    item_error_row = error_spread_df[
        error_spread_df["item_id"] == item_id
    ]

    if len(item_error_row) == 0:
        raise ValueError(
            f"Invalid item_id: '{item_id}' not found in item_error_spread."
        )

    typical_daily_error = float(
        item_error_row["typical_daily_error"].values[0]
    )

    item_group = str(item_error_row["group"].values[0])

    # 4. Calculate safety stock
    safety_stock = (
        1.65 * typical_daily_error * math.sqrt(lead_time_days)
    )

    # 5. Calculate reorder point
    reorder_point = expected_demand_lead_time + safety_stock

    # 6. Calculate days of stock left
    if avg_daily_forecast > 0:
        days_of_stock_left = current_stock / avg_daily_forecast
    else:
        days_of_stock_left = 999.0 if current_stock > 0 else 0.0

    # 7. Calculate order quantity
    total_window_days = lead_time_days + review_days

    if len(daily_forecasts) >= total_window_days:
        expected_demand_window = float(
            np.sum(daily_forecasts[:total_window_days])
        )
    else:
        missing_days = total_window_days - len(daily_forecasts)

        expected_demand_window = (
            float(np.sum(daily_forecasts))
            + missing_days * avg_daily_forecast
        )

    raw_order_qty = (
        expected_demand_window + safety_stock - current_stock
    )

    order_quantity = max(0, math.ceil(raw_order_qty))

    # 8. Risk classification
    if days_of_stock_left < lead_time_days:
        risk = "High"
    elif days_of_stock_left < (lead_time_days + 3):
        risk = "Medium"
    else:
        risk = "Low"

    return {
        "item_id": item_id,
        "group": item_group,
        "forecast_model_used": forecast_source,
        "current_stock": current_stock,
        "lead_time_days": lead_time_days,
        "average_daily_forecast": round(avg_daily_forecast, 2),
        "expected_demand_in_lead_time": round(expected_demand_lead_time, 2),
        "typical_daily_error": round(typical_daily_error, 2),
        "safety_stock": round(safety_stock, 2),
        "reorder_point": round(reorder_point, 2),
        "days_of_stock_left": round(days_of_stock_left, 1),
        "order_quantity": int(order_quantity),
        "risk": risk
    }


# Backward-compatible alias
calculate_reorder_plan = reorder_advice
'''


# 3. Save updated reorder.py
with open(reorder_py_path, "w") as f:
    f.write(updated_reorder_code)

print(f"[SAVED] Updated: {reorder_py_path}")


# 4. Import and reload module
if HANDOVER_DIR not in sys.path:
    sys.path.insert(0, HANDOVER_DIR)

import reorder
importlib.reload(reorder)

from reorder import reorder_advice, calculate_reorder_plan

print(
    "[OK] Successfully imported reorder_advice "
    "and calculate_reorder_plan!\n"
)


# 5. Run all 6 verification tests
sample_item = item_error_spread["item_id"].iloc[0]

print("--- Running Verification Tests ---")

# Test 1: Normal reorder case
res_normal = reorder_advice(
    sample_item,
    current_stock=5,
    lead_time_days=5,
    h7_preds_df=h7_val_preds,
    h28_preds_df=h28_val_preds,
    error_spread_df=item_error_spread
)

print(
    f"[PASS] Test 1 (Normal Case): "
    f"model={res_normal['forecast_model_used']}, "
    f"order_qty={res_normal['order_quantity']}, "
    f"risk={res_normal['risk']}"
)


# Test 2: Zero stock
res_zero_stock = reorder_advice(
    sample_item,
    current_stock=0,
    lead_time_days=5,
    h7_preds_df=h7_val_preds,
    h28_preds_df=h28_val_preds,
    error_spread_df=item_error_spread
)

print(
    f"[PASS] Test 2 (Zero Stock): "
    f"days_left={res_zero_stock['days_of_stock_left']}, "
    f"order_qty={res_zero_stock['order_quantity']}, "
    f"risk={res_zero_stock['risk']}"
)


# Test 3: Zero demand
zero_demand_df = pd.DataFrame({
    "item_id": [sample_item] * 14,
    "date": pd.date_range("2016-03-28", periods=14),
    "forecast": [0.0] * 14
})

res_zero_demand = reorder_advice(
    sample_item,
    current_stock=10,
    lead_time_days=5,
    h7_preds_df=zero_demand_df,
    h28_preds_df=zero_demand_df,
    error_spread_df=item_error_spread
)

print(
    f"[PASS] Test 3 (Zero Demand): "
    f"days_left={res_zero_demand['days_of_stock_left']}, "
    f"order_qty={res_zero_demand['order_quantity']}, "
    f"risk={res_zero_demand['risk']}"
)


# Test 4: Negative stock
try:
    reorder_advice(
        sample_item,
        current_stock=-5,
        lead_time_days=5,
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )
    print("[FAIL] Test 4: Negative stock was not rejected!")

except ValueError as e:
    print(f"[PASS] Test 4 (Negative Stock): {e}")


# Test 5: Zero lead time
try:
    reorder_advice(
        sample_item,
        current_stock=10,
        lead_time_days=0,
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )
    print("[FAIL] Test 5: Zero lead time was not rejected!")

except ValueError as e:
    print(f"[PASS] Test 5 (Zero Lead Time): {e}")


# Test 6: Invalid item ID
try:
    reorder_advice(
        "INVALID_ITEM_999",
        current_stock=10,
        lead_time_days=5,
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )
    print("[FAIL] Test 6: Invalid item was not rejected!")

except ValueError as e:
    print(f"[PASS] Test 6 (Invalid Item): {e}")


# 6. Final verification
print("\n--- Final File Verification ---")

for filename in os.listdir(HANDOVER_DIR):
    filepath = os.path.join(HANDOVER_DIR, filename)

    if os.path.isfile(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"[SAVED] {filename:<30} {size_kb:.2f} KB")

print("\nVerification execution completed.")


[SAVED] Updated: /content/backend_handover/reorder.py
[OK] Successfully imported reorder_advice and calculate_reorder_plan!

--- Running Verification Tests ---
[PASS] Test 1 (Normal Case): model=H7, order_qty=12, risk=Medium
[PASS] Test 2 (Zero Stock): days_left=0.0, order_qty=17, risk=High
[PASS] Test 3 (Zero Demand): days_left=999.0, order_qty=0, risk=Low
[PASS] Test 4 (Negative Stock): Invalid current_stock (-5): must be >= 0.
[PASS] Test 5 (Zero Lead Time): Invalid lead_time_days (0): must be > 0.
[PASS] Test 6 (Invalid Item): Invalid item_id: 'INVALID_ITEM_999' not found in H7 forecast data.

--- Final File Verification ---
[SAVED] item_error_spread.csv          221.30 KB
[SAVED] reorder.py                     3.94 KB

Verification execution completed.
